In [ ]:
# create new environment (here neurom)
# git clone https://github.com/netneurolab/neuromaps.git
# pip install git+https://github.com/netneurolab/netneurotools.git

In [ ]:
import neuromaps
from scipy.io import savemat
import scipy
#import abagen
import numpy as np
import pandas as pd
import nibabel as nib
# from sklearn import decomposition
from neuromaps import datasets,resampling
from neuromaps.parcellate import Parcellater
from nibabel.freesurfer.io import read_annot
from neuromaps.datasets import fetch_annotation
from netneurotools import datasets as nntdata
from neuromaps.images import dlabel_to_gifti

In [ ]:
# Set directories ####################################
######################################################
repo_dir = '/Users/Christina/sciebo/Aging_multiv/'
parcellationDir = repo_dir + 'scripts/Data/'

# load data
labels = (parcellationDir + 'fslr32k/' +
            'Schaefer2018_200Parcels_7Networks_order.dlabel.nii')
#rhlabels = (parcellationDir + 'fslr32k/' +
#            'Schaefer2018_100Parcels_7Networks_order_rh.label.gii')
labelinfo = np.loadtxt(parcellationDir + 'fslr32k/' +
                        'Schaefer2018_200Parcels_7Networks_order_info.txt',
                        dtype='str', delimiter='\t')

In [ ]:
# get all annotations that are in surface space already and only need parcellation to the schaefer 200 atlas 
############################################################################################################

ANNOTATIONS = [
    ('abagen', 'genepc1', 'fsaverage', '10k'),
    ('hcps1200', 'myelinmap', 'fsLR', '32k'),
    ('hcps1200', 'thickness', 'fsLR', '32k'),
    ('hill2010', 'devexp', 'fsLR', '164k'),
    ('hill2010', 'evoexp', 'fsLR', '164k'),
    ('raichle', 'cbf', 'fsLR', '164k'),
    ('raichle', 'cbv', 'fsLR', '164k'),
    ('raichle', 'cmr02', 'fsLR', '164k'),
    ('raichle', 'cmrglc', 'fsLR', '164k'),
    ('reardon2018', 'scalingnih', 'civet', '41k'),
    ('reardon2018', 'scalingpnc', 'civet', '41k'),
    ('margulies2016', 'fcgradient01', 'fsLR', '32k'),
    ('margulies2016', 'fcgradient02', 'fsLR', '32k'),
    ('margulies2016', 'fcgradient03', 'fsLR', '32k'),
    ('margulies2016', 'fcgradient04', 'fsLR', '32k'),
    ('margulies2016', 'fcgradient05', 'fsLR', '32k'),
    ('margulies2016', 'fcgradient06', 'fsLR', '32k'),
    ('margulies2016', 'fcgradient07', 'fsLR', '32k'),
    ('margulies2016', 'fcgradient08', 'fsLR', '32k'),
    ('margulies2016', 'fcgradient09', 'fsLR', '32k'),
    ('margulies2016', 'fcgradient10', 'fsLR', '32k'),
    ('mueller2013', 'intersubjvar', 'fsLR', '164k'),
    ('sydnor2021', 'SAaxis', 'fsLR', '32k')
]

# prepare maps
neuromaps_microsc = []

for (src, desc, space, den) in ANNOTATIONS:
    target = datasets.fetch_annotation(source=src, desc=desc, space=space, den=den)
    if src=='hill2010':
        hemi = 'R'
    else:
        hemi = None

    # this won't be used later, just so that we have both src and trg inputs
    tempmap = datasets.fetch_annotation(desc='myelinmap')

    if not(space=='fsLR' and den=='32k'):
        tempmap_rs, target_rs = resampling.resample_images(src=tempmap, trg=target,
                                                           src_space='fsLR',
                                                           trg_space=space,
                                                           hemi=hemi,
                                                           method='linear',
                                                           resampling='transform_to_alt',
                                                           alt_spec=('fslr', '32k'))
    else:
        tempmap_rs = tempmap
        target_rs = target

    if hemi=='R':
        # mirror if only one hemisphere
        parcellation = labels #[lhlabels, rhlabels]
        parc = Parcellater(dlabel_to_gifti(schaefer), 'fsLR')
        parcellated_target = parc.fit_transform((target_rs[0], target_rs[0]),
                                                 'fslr')
        #parcellated_target = scipy.stats.zscore(parcellated_target)

    else:
        parcellation = labels #[lhlabels, rhlabels]
        schaefer = nntdata.fetch_schaefer2018('fslr32k')['200Parcels7Networks']
        parc = Parcellater(dlabel_to_gifti(schaefer), 'fsLR')
        parcellated_target = parc.fit_transform(target_rs, 'fsLR')
        #parcellated_target = scipy.stats.zscore(parcellated_target)

    neuromaps_microsc.append(parcellated_target)

    print('\nMap: %s done!' % desc)

neuromaps_microsc = np.array(neuromaps_microsc).T
microsclabels = [f"{item[0]}-{item[1]}" for item in ANNOTATIONS]


savemat('surface_maps_schaefer200.mat', {'neuromaps_mic': neuromaps_microsc, 'microsclabels': microsclabels})

In [ ]:
# Get all volumetric maps and parcellate to 200 Schaefer parcels => PET data should remain in volumetric space
##############################################################################################################

# Reason: Partial volume effects (PVE) in PET images are increased when volumetric data is transformed to the surface, 
# resulting in maps that are heavily biased by the underlying curvature of the surface. For this reason, it is generally 
# not recommended to transform PET volumes to the surface. (https://netneurolab.github.io/neuromaps/user_guide/transformations.html#parcellations)
# pip install git+https://github.com/netneurolab/netneurotools.git

from neuromaps.parcellate import Parcellater
from nilearn.datasets import fetch_atlas_schaefer_2018

# # Example code to directly load and parcellate with neuromaps (provided by Shafiei et al.)
#mni_atlas = fetch_atlas_schaefer_2018(n_rois=200)['maps']
#parcellater = Parcellater(mni_atlas, 'MNI152')
#orig_map = datasets.fetch_annotation(desc='way100635')
#parcellated = parcellater.fit_transform(orig_map, 'MNI152', True)
#savemat('mni_pet.mat', {'mni_pet': parcellated})

In [ ]:
# Get PET/receptor data - only take volumetric maps here
#########################################################

ANNOTATIONS = [
    ('aghourian2017', 'feobv', 'MNI152', '1mm'),
    ('alarkurtti2015', 'raclopride', 'MNI152', '3mm'),
    ('bedard2019', 'feobv', 'MNI152', '1mm'),
    ('beliveau2017', 'az10419369', 'MNI152', '1mm'),
    #('beliveau2017', 'az10419369', 'fsaverage', '164k'),
    ('beliveau2017', 'cimbi36', 'MNI152', '1mm'),
    #('beliveau2017', 'cimbi36', 'fsaverage', '164k'),
    ('beliveau2017', 'cumi101', 'MNI152', '1mm'),
    #('beliveau2017', 'cumi101', 'fsaverage', '164k'),
    ('beliveau2017', 'dasb', 'MNI152', '1mm'),
    #('beliveau2017', 'dasb', 'fsaverage', '164k'),
    ('beliveau2017', 'sb207145', 'MNI152', '1mm'),
    #('beliveau2017', 'sb207145', 'fsaverage', '164k'),
    ('ding2010', 'mrb', 'MNI152', '1mm'),
    ('dubois2015', 'abp688', 'MNI152', '1mm'),
    ('dukart2018', 'flumazenil', 'MNI152', '3mm'),
    ('dukart2018', 'fpcit', 'MNI152', '3mm'),
    ('fazio2016', 'madam', 'MNI152', '3mm'),
    ('finnema2016', 'ucbj', 'MNI152', '1mm'),
    ('gallezot2010', 'p943', 'MNI152', '1mm'),
    ('gallezot2017', 'gsk189254', 'MNI152', '1mm'),
    ('galovic2021', 'ge179', 'MNI152', '1mm'),
    ('hesse2017', 'methylreboxetine', 'MNI152', '3mm'),
    ('hillmer2016', 'flubatine', 'MNI152', '1mm'),
    ('jaworska2020', 'fallypride', 'MNI152', '1mm'),
    ('kaller2017', 'sch23390', 'MNI152', '3mm'),
    ('kantonen2020', 'carfentanil', 'MNI152', '3mm'),
    ('kim2020', 'ps13', 'MNI152', '2mm'),
    ('laurikainen2018', 'fmpepd2', 'MNI152', '1mm'),
    ('lois2018', 'pbr28', 'MNI152', '2mm'),
    ('lukow2022', 'ro154513', 'MNI152', '2mm'),
    ('malen2022', 'raclopride', 'MNI152', '2mm'),
    ('naganawa2020', 'lsn3172176', 'MNI152', '1mm'),
    ('neurosynth', 'cogpc1', 'MNI152', '2mm'),
    ('norgaard2021', 'flumazenil', 'MNI152', '1mm'),
    #('norgaard2021', 'flumazenil', 'fsaverage', '164k'),
    ('normandin2015', 'omar', 'MNI152', '1mm'),
    ('radnakrishnan2018', 'gsk215083', 'MNI152', '1mm'),
    ('rosaneto', 'abp688', 'MNI152', '1mm'),
    ('sandiego2015', 'flb457', 'MNI152', '1mm'),
    ('sasaki2012', 'fepe2i', 'MNI152', '1mm'),
    ('satterthwaite2014', 'meancbf', 'MNI152', '1mm'),
    ('savli2012', 'altanserin', 'MNI152', '3mm'),
    ('savli2012', 'dasb', 'MNI152', '3mm'),
    ('savli2012', 'p943', 'MNI152', '3mm'),
    ('savli2012', 'way100635', 'MNI152', '3mm'),
    ('smart2019', 'abp688', 'MNI152', '1mm'),
    ('smith2017', 'flb457', 'MNI152', '1mm'),
    ('tuominen', 'feobv', 'MNI152', '2mm'),
    ('turtonen2020', 'carfentanil', 'MNI152', '1mm'),
    ('vijay2018', 'ly2795050', 'MNI152', '2mm'),
    ('wey2016', 'martinostat', 'MNI152', '2mm')
]


mni_atlas = fetch_atlas_schaefer_2018(n_rois=200)['maps']
parcellater = Parcellater(mni_atlas, 'MNI152')

# prepare maps
neuromaps_rec = []

for (src, desc, space, den) in ANNOTATIONS:
    orig = datasets.fetch_annotation(source=src, desc=desc, space=space, den=den)
    parcellated = parcellater.fit_transform(orig, 'MNI152', True)
    #print(f"Data before z-scoring (desc={desc}): Min={np.nanmin(parcellated)}, Max={np.nanmax(parcellated)}, NaNs={np.isnan(parcellated).sum()}")
    #if np.nanstd(parcellated) == 0:
    #   print(f"Data for {desc} is constant or empty. Skipping z-scoring.")
    #parcellated = scipy.stats.zscore(parcellated, nan_policy='omit')
    neuromaps_rec.append(parcellated)


neuromaps_receptors= np.array(neuromaps_rec).T
reclabels = [f"{item[0]}-{item[1]}" for item in ANNOTATIONS]

savemat('rec_maps_schaefer200.mat', {'neuromaps_receptors': neuromaps_receptors, 'reclabels': reclabels})

In [ ]:
# BigBrain data NOT DONE YET - need to get the respective files....
#####################################################################
#  Get BigBrain data - code by Shafiei et al.
# cortical layers
bigbrainwarppath = '/home/gshafiei/data1/Projects/packages/BigBrainWarp/spaces/'
schaefer_fsaverage_lh = read_annot(parcellationDir + 'FreeSurfer5.3/' +
                                   'fsaverage/label/' +
                                   'lh.Schaefer2018_100Parcels_7Networks' +
                                   '_order.annot')
schaefer_fsaverage_rh = read_annot(parcellationDir + 'FreeSurfer5.3/' +
                                   'fsaverage/label/' +
                                   'rh.Schaefer2018_100Parcels_7Networks' +
                                   '_order.annot')

parcellatedData_all = np.zeros((6, 100))
for i in range(6):
    lh_file = (bigbrainwarppath + 'tpl-fsaverage/' +
            'tpl-fsaverage_hemi-L_den-164k_desc-layer%s_thickness.shape.gii' % str(i+1))
    rh_file = (bigbrainwarppath + 'tpl-fsaverage/' +
            'tpl-fsaverage_hemi-R_den-164k_desc-layer%s_thickness.shape.gii' % str(i+1))

    data_lh = nib.load(lh_file).darrays[0].data
    data_rh = nib.load(rh_file).darrays[0].data

    parcelIDs_lh = np.unique(schaefer_fsaverage_lh[0])
    parcelIDs_lh = np.delete(parcelIDs_lh, 0)

    parcellatedData_lh = np.zeros((1, len(parcelIDs_lh)))

    for IDnum in parcelIDs_lh:
        idx = np.where(schaefer_fsaverage_lh[0] == IDnum)[0]
        parcellatedData_lh[:, IDnum-1] = np.nanmean(data_lh[idx])


    parcelIDs_rh = np.unique(schaefer_fsaverage_rh[0])
    parcelIDs_rh = np.delete(parcelIDs_rh, 0)

    parcellatedData_rh = np.zeros((1, len(parcelIDs_rh)))

    for IDnum in parcelIDs_rh:
        idx = np.where(schaefer_fsaverage_rh[0] == IDnum)[0]
        parcellatedData_rh[:, IDnum-1] = np.nanmean(data_rh[idx])

    parcellatedData = np.hstack((parcellatedData_lh, parcellatedData_rh))

    parcellatedData_all[i, :] = parcellatedData

layer_data = parcellatedData_all.T
all_labels.extend(list(np.arange(6)+1))
all_data = np.hstack((receptor_data, celltype_exp, layer_data))